In [21]:
# Imports
import os
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
import joblib


In [22]:
# 1) Load data (same categories / splits as Lab6.4)
categories = ['alt.atheism', 'comp.graphics', 'sci.med', 'sci.space']
train_raw = fetch_20newsgroups(subset='train', remove=('headers','footers','quotes'),
                              categories=categories, random_state=42)
test_raw  = fetch_20newsgroups(subset='test',  remove=('headers','footers','quotes'),
                              categories=categories, random_state=42)

train_df = pd.DataFrame({'text': train_raw.data, 'labels': train_raw.target})
test_df  = pd.DataFrame({'text': test_raw.data,  'labels': test_raw.target})

# 2) create dev set (10% of train), stratified
train_df, dev_df = train_test_split(train_df, test_size=0.1, random_state=0,
                                    stratify=train_df[['labels']])
print(f"Sizes: train={len(train_df)}, dev={len(dev_df)}, test={len(test_df)}")

# 3) Conventional baseline: TF\-IDF + LinearSVC
svc_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_df=0.9, min_df=3, ngram_range=(1,2), max_features=20000)),
    ('svc', LinearSVC(C=1.0, max_iter=20000))
])
svc_pipeline.fit(train_df['text'], train_df['labels'])
svc_preds = svc_pipeline.predict(test_df['text'])
print("\n=== TF\-IDF + LinearSVC ===")
print(classification_report(test_df['labels'], svc_preds, digits=4, target_names=categories))

# save pipeline and report
joblib.dump(svc_pipeline, 'tfidf_linear_svc.joblib')
svc_report = classification_report(test_df['labels'], svc_preds, output_dict=True, target_names=categories)
pd.DataFrame(svc_report).transpose().to_csv('tfidf_linear_svc_report.csv')

# 4) Try Transformer fine-tune (DistilRoBERTa) if available
try:
    from simpletransformers.classification import ClassificationModel, ClassificationArgs

    print("\nStarting DistilRoBERTa fine-tune (this may take a while)...")
    # prepare dataframes expected by simpletransformers
    train_small = train_df[['text','labels']].rename(columns={'text':'text','labels':'labels'})
    dev_small   = dev_df[['text','labels']].rename(columns={'text':'text','labels':'labels'})
    test_texts  = test_df['text'].tolist()

    model_args = ClassificationArgs()
    model_args.num_train_epochs = 3
    model_args.learning_rate = 2e-5
    model_args.train_batch_size = 16
    model_args.max_seq_length = 256
    model_args.overwrite_output_dir = True
    model_args.evaluate_during_training = True
    model_args.use_early_stopping = True
    model_args.early_stopping_patience = 2
    model_args.early_stopping_delta = 0.01
    model_args.manual_seed = 0
    model_args.use_multiprocessing = False
    model = ClassificationModel('roberta', 'distilroberta-base', num_labels=4,
                                args=model_args, use_cuda=False)
    model.train_model(train_small, eval_df=dev_small)
    preds, probs = model.predict(test_texts)
    print("\n=== DistilRoBERTa (fine-tuned) ===")
    print(classification_report(test_df['labels'], preds, digits=4, target_names=categories))
    pd.DataFrame(classification_report(test_df['labels'], preds, output_dict=True, target_names=categories)).transpose().to_csv('distilroberta_report.csv')
except Exception as e:
    print("\nSkipping DistilRoBERTa fine-tune: simpletransformers not available or error.")
    print("Reason:", e)
    print("If you want to run it, install simpletransformers in a suitable environment.")

# 5) Build comparison table from available CSV reports
files = {
    "TF-IDF+LinearSVC": "tfidf_linear_svc_report.csv",
    "DistilRoBERTa": "distilroberta_report.csv",
    "BERT_lab6": "bert_report.csv"  # optional: export from Lab6.4 as this filename
}

def load_report(path):
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path, index_col=0)
    # standardize column names if necessary
    # prefer columns named 'precision','recall' and any 'f1' column
    cols = []
    for c in ("precision","recall"):
        if c in df.columns:
            cols.append(c)
    f1_col = next((c for c in df.columns if "f1" in c.lower()), None)
    if f1_col:
        cols.append(f1_col)
    if not cols:
        # try common alternative names
        for alt in ("precision","recall","f1-score"):
            if alt in df.columns:
                cols.append(alt)
    return df[cols] if cols else df

reports = {}
for name, path in files.items():
    r = load_report(path)
    if r is None:
        print(f"Missing report: {path} (skipping {name})")
    else:
        reports[name] = r

if not reports:
    print("No reports found besides the TF\-IDF baseline. Run other models and save reports as CSVs.")
else:
    # collect all row labels
    rows = []
    for r in reports.values():
        rows.extend(r.index.tolist())
    rows = list(dict.fromkeys(rows))  # preserve order unique
    # build comparison
    comp = []
    for label in rows:
        row = {"label": label}
        for name, r in reports.items():
            if label in r.index:
                p = r.loc[label].get("precision") if "precision" in r.columns else None
                rec = r.loc[label].get("recall") if "recall" in r.columns else None
                f1 = None
                f1_candidates = [c for c in r.columns if "f1" in c.lower()]
                if f1_candidates:
                    f1 = r.loc[label].get(f1_candidates[0])
                row[f"{name} P"] = float(p) if p is not None else None
                row[f"{name} R"] = float(rec) if rec is not None else None
                row[f"{name} F1"] = float(f1) if f1 is not None else None
            else:
                row[f"{name} P"] = None
                row[f"{name} R"] = None
                row[f"{name} F1"] = None
        comp.append(row)
    comp_df = pd.DataFrame(comp).set_index("label")
    print("\nComparison (Precision / Recall / F1):\n")
    print(comp_df.round(4).to_string())
    comp_df.to_csv("comparison_report.csv")
    print("\nSaved `comparison_report.csv`.")

Sizes: train=2025, dev=226, test=1498

=== TF\-IDF + LinearSVC ===
               precision    recall  f1-score   support

  alt.atheism     0.8438    0.7618    0.8007       319
comp.graphics     0.7684    0.8869    0.8234       389
      sci.med     0.8654    0.7955    0.8289       396
    sci.space     0.7909    0.7970    0.7939       394

     accuracy                         0.8124      1498
    macro avg     0.8171    0.8103    0.8117      1498
 weighted avg     0.8160    0.8124    0.8123      1498


Skipping DistilRoBERTa fine-tune: simpletransformers not available or error.
Reason: cannot import name 'is_remote_url' from 'transformers.utils' (C:\Users\stolw\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\utils\__init__.py)
If you want to run it, install simpletransformers in a suitable environment.
Missing report: distilroberta_report.csv (skipping DistilRoBERTa)
Missing report: bert_report.csv (skipping BERT_lab6)

Comparison (Precision / Recall / F1):

 